In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spaceship-titanic/sample_submission.csv
/kaggle/input/spaceship-titanic/train.csv
/kaggle/input/spaceship-titanic/test.csv


# **1. Loading modules:-**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,StackingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from catboost import CatBoostClassifier

# **2. Loading source file:-**

In [3]:
train=pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')

#-----------------------------------------------------------------

test=pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

#----------------------------------------------------------------

submission=pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')

# **3.Dropping unnecessary columns:-**

In [4]:
train.drop(['PassengerId','Name'],axis=1,inplace=True)
#--------------------------------------------------------
test.drop(['PassengerId','Name'],axis=1,inplace=True)

# **4.Data Cleaning:-**

**4.2. Treating continuous values:-**

In [5]:
reg=['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','Age']

train[reg]=train[reg]+3
for i in reg:
    train[i]=np.log(train[i])
    
#--------------------------------------

test[reg]=test[reg]+3
for i in reg:
    test[i]=np.log(test[i])

**4.3. Extracting cabin columns:-**

In [6]:
train['Cabin'].fillna('0',inplace=True)
cabin=list(train['Cabin'])

train['cabin_group1']=[i.split('/')[0] if i!='0' else '0' for i in cabin]
train['cabin_group2']=[i.split('/')[2] if i!='0' else '0' for i in cabin]
      

train['Cabin'].replace({'0',np.nan},inplace=True)
train.drop(['Cabin'],axis=1,inplace=True)

#--------------------------------------------------------------------------

test['Cabin'].fillna('0',inplace=True)
cabin=list(test['Cabin'])

test['cabin_group1']=[i.split('/')[0] if i!='0' else '0' for i in cabin]
test['cabin_group2']=[i.split('/')[2] if i!='0' else '0' for i in cabin]
      

test['Cabin'].replace({'0',np.nan},inplace=True)
test.drop(['Cabin'],axis=1,inplace=True)


**Label Encoding**

In [7]:
col=['HomePlanet','CryoSleep','Destination','VIP','Transported','cabin_group1','cabin_group2']

encode=LabelEncoder()
for i in col:
    encode.fit(train[i])
    train[i]=encode.transform(train[i])
    
#-----------------------------------------------------

col=['HomePlanet','CryoSleep','Destination','VIP','cabin_group1','cabin_group2']
encode=LabelEncoder()
for i in col:
    encode.fit(test[i])
    test[i]=encode.transform(test[i])

**4.4. Imputation:-**

In [8]:
impute=KNNImputer(n_neighbors=6)
impute.fit(train)
train=pd.DataFrame(impute.transform(train),columns=train.columns)

#----------------------------------------------------------------

impute=KNNImputer(n_neighbors=6)
impute.fit(test)
test=pd.DataFrame(impute.transform(test),columns=test.columns)

# **5. Modelling**

In [9]:
x=train.drop(['Transported'],axis=1)
y=train['Transported']

**Algorithm**

In [10]:
models=[]
models.append(('Logistic Regression',LogisticRegression()))
models.append(('SVC',SVC()))
models.append(('Random Forest',RandomForestClassifier()))
models.append(('Gradient',GradientBoostingClassifier()))
models.append(('LGBM',LGBMClassifier()))
models.append(('XGB',XGBClassifier()))


In [11]:
for name,model in models:
    score=cross_val_score(model,x,y,scoring='accuracy',cv=10,n_jobs=-1)
    print(name,np.mean(score))

Logistic Regression 0.7787909209951986
SVC 0.7968506540745738
Random Forest 0.7960485695012103
Gradient 0.7997289790087694
LGBM 0.8032956364165443
XGB 0.7985794214515296


In [12]:
stacking=StackingClassifier(models,cv=10)
stacking.fit(x,y)
submission['Transported']=stacking.predict(test)

In [13]:
submission.to_csv('ver1.csv',index=False)

In [14]:
submission

,PassengerId,Transported
0,0013_01,1.0
1,0018_01,0.0
2,0019_01,1.0
3,0021_01,1.0
4,0023_01,1.0
...,...,...
4272,9266_02,1.0
4273,9269_01,0.0
4274,9271_01,1.0
4275,9273_01,1.0
